# TabPFN-Wide attention smoke test

First real attention run. Check the query invariance control before interpreting rankings.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from stablewide.notebook_utils import find_project_root, run_experiment, read_outputs, core_subsample_table
ROOT = find_project_root(ROOT)
print(ROOT)

C:\Users\scoti\PycharmProjects\STABLE_WIDE_FINAL


In [2]:
try:
    import tabpfnwide
    print("tabpfnwide available")
except Exception as e:
    raise RuntimeError("Install tabpfnwide before this notebook: pip install tabpfnwide") from e

tabpfnwide available


In [3]:
OUT = ROOT / "outputs" / "02_attention_smoke"
args = [
    "--synthetic", "--quick",
    "--methods", "anova,rf,l1,l2,attention",
    "--out", str(OUT),
]
run_experiment(ROOT, args)

C:\Users\scoti\PycharmProjects\STABLE_WIDE_FINAL\.venv\Scripts\python.exe C:\Users\scoti\PycharmProjects\STABLE_WIDE_FINAL\run_experiment.py --synthetic --quick --methods anova,rf,l1,l2,attention --out C:\Users\scoti\PycharmProjects\STABLE_WIDE_FINAL\outputs\02_attention_smoke


CompletedProcess(args=['C:\\Users\\scoti\\PycharmProjects\\STABLE_WIDE_FINAL\\.venv\\Scripts\\python.exe', 'C:\\Users\\scoti\\PycharmProjects\\STABLE_WIDE_FINAL\\run_experiment.py', '--synthetic', '--quick', '--methods', 'anova,rf,l1,l2,attention', '--out', 'C:\\Users\\scoti\\PycharmProjects\\STABLE_WIDE_FINAL\\outputs\\02_attention_smoke'], returncode=0)

In [4]:
res = read_outputs(OUT)
print(res.get("decision", "No decision file"))

=== Pre-registered pilot decision (v4) ===
decision k=10; AUROC floor=0.75; 
gene-stability ceiling=0.50; pairwise margin=0.10
minimum outer splits with finite estimates=3
Insufficient outer-split evidence for a decision; do not interpret as a null result.

INCONCLUSIVE


In [5]:
df = res["results_outer.csv"]
cols = [c for c in ["outer_id","method","experiment","n_train","k","jaccard_gene","jaccard_module","auroc_mean","pred_mad","pred_max_abs","query_invariance_pass"] if c in df.columns]
df[df["method"].eq("attention")][cols].round(4)

,outer_id,method,experiment,n_train,k,jaccard_gene,jaccard_module,auroc_mean,pred_mad,pred_max_abs,query_invariance_pass
4,0,attention,seed,352,5,0.6190,0.8333,0.8633,0.0758,0.4871,NaN
5,0,attention,seed,352,10,0.7677,1.0000,0.8633,0.0758,0.4871,NaN
6,0,attention,seed,352,20,1.0000,1.0000,0.8633,0.0758,0.4871,NaN
7,0,attention,seed,352,50,0.5633,0.4872,0.8633,0.0758,0.4871,NaN
8,0,attention,query,352,5,1.0000,1.0000,0.8542,0.0000,0.0000,True
9,0,attention,query,352,10,1.0000,1.0000,0.8542,0.0000,0.0000,True
10,0,attention,query,352,20,1.0000,1.0000,0.8542,0.0000,0.0000,True
11,0,attention,query,352,50,1.0000,1.0000,0.8542,0.0000,0.0000,True
28,0,attention,subsample,100,5,0.1706,0.4317,0.7826,0.1003,0.5988,NaN
29,0,attention,subsample,100,10,0.3286,0.5688,0.7826,0.1003,0.5988,NaN


For the query control, anchor predictions should stay numerically unchanged. Any attention change is interpreted only as a change in the aggregate readout.